In [21]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders.text import TextLoader
from langchain_community.vectorstores import SKLearnVectorStore
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [ ]:
f = open(r'C:\Users\joseantonio.clemente\Documents\Langchain\api_key.txt')
api_key = f.read()

In [3]:
loader = TextLoader(file_path=r"C:\Users\joseantonio.clemente\Documents\Langchain\Práctica FInal\Información_computadoras.txt")

In [4]:
data = loader.load()

In [7]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
docs = text_splitter.split_documents(data)

In [11]:
embed = OpenAIEmbeddings(openai_api_key=api_key)

In [10]:
%pwd

'c:\\Users\\joseantonio.clemente\\Documents\\Langchain\\Práctica FInal'

In [ ]:
persist_path="./computadoras_embedding_db"

vector_store = SKLearnVectorStore.from_documents(
    documents=docs,
    embedding=embed,
    persist_path=persist_path,
    serializer="parquet",
)

In [15]:
vector_store.persist()

In [17]:
llm = ChatOpenAI(temperature=0,openai_api_key=api_key)
compressor = LLMChainExtractor.from_llm(llm)

In [18]:
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=vector_store.as_retriever())

In [19]:
compressed_docs = compression_retriever.invoke("¿qué es el sistema operativo?")

In [20]:
compressed_docs[0].page_content

'El sistema operativo es el programa que gestiona y administra todos los recursos del ordenador, controla, por ejemplo, qué programas se ejecutan y cuándo, administra la memoria y los accesos a los dispositivos E/S, provee las interfases entre dispositivos, incluso entre el computador y el usuario.'